In [ ]:
# =============================================================================
# jobs/34_build_exposure_grid.py
# The 100 m exposure grid: address points added up into cells, joined to the flood band shares of each cell, with the critical sites
# and an adjustable composite score (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Restricted schema: it is built from address-level data.
#
# INPUTS  config.TBL_ADDRESS_POINTS   job 33   people, households, over-65s, deprivation and the flood band at each address
#         config.TBL_HAZARD_CELLS     job 30   the share of each cell's area in each band (rivers and sea, Flood Zones split coastal/fluvial)
#         config.TBL_CRITICAL_SITES   job 31   optional; if the table does not exist the critical site columns are zero
# OUTPUT  config.TBL_EXPOSURE_GRID    one row per cell that has people, a flood band or a critical site, with
#           cell_key, easting, northing (south-west corner), tile
#           people, households, over65, n_points (addresses), imd_score, imd_decile (people-weighted)
#           people_rs_<band>, households_rs_<band>      people and households by rivers and sea band
#           people_fz2, people_fz3, people_fz3_sea, people_fz3_river, people_fz3_both    people by Flood Zone and mapping source
#           share_rs_<band>, share_fz2, share_fz3, share_fz3_sea, share_fz3_river, share_fz3_both    share of the cell's AREA in each band
#           haz_rs          area-weighted rivers and sea band, 0 to 4 (a quarter High, rest dry is 1.0, not 4.0)
#           critical_sites, critical_health, critical_education, critical_emergency, critical_utilities, critical_in_rs_high_medium
#           pop_pct         national population percentile among populated cells
#           risk_score      0 to 100, from config.RISK_SCORE_WEIGHTS; rivers and sea only (config.RISK_SCORE_INCLUDE_SURFACE is False)
# The layers are always stored. The score is a ranking aid made from them, and the weights can be changed here and the job re-run.
# No surface water: warning areas are not issued for it. Coastal, fluvial and surface water are different hazards and are not merged.
#
# HOW  Each 50 km tile is built on its own (points to cells, join shares and critical sites). The people of every cell are then gathered
# (one column) to make the population percentile national, and each tile is scored and written on its own. The whole grid, several million
# rows by about 55 columns, is never joined into one frame: doing so exhausted the driver. Overwritten in full each run.
#
# WHAT TO CHECK (printed): people in the grid against people in the address points (equal); people by rivers and sea band against the
# address points; cells and score distribution; the highest-scoring cells (do they look like places you would expect?).
#
# Widgets: tiles, workers, use_critical.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

# shapely is not on every cluster. If it has to be installed the notebook restarts Python: run all again once.
try:
    import shapely  # noqa: F401
except ImportError:
    get_ipython().run_line_magic("pip", "install -q shapely")
    dbutils.library.restartPython()

import importlib
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

import config
import utils.grid100 as _g
import utils.grid_build as _gb
importlib.reload(_g)
importlib.reload(_gb)
from utils.grid_build import add_score, build_grid, critical_to_cells, points_to_cells, shares_wide
from utils.hazard_raster import tile_code_at
from utils.helpers import get_spark

spark = get_spark()

dbutils.widgets.text("tiles", "", "Only these tiles, for a trial (comma separated). Blank is all. A trial does not write the table")
dbutils.widgets.text("workers", "1", "Tiles to run at once (8 on a 16 core node)")
dbutils.widgets.dropdown("use_critical", "yes", ["yes", "no"], "Count critical sites if the table exists")
ONLY = [t.strip().upper() for t in dbutils.widgets.get("tiles").split(",") if t.strip()]
WORKERS = max(1, int(dbutils.widgets.get("workers") or 1))
USE_CRITICAL = dbutils.widgets.get("use_critical") == "yes"
print(f"Workers {WORKERS}. Trial tiles: {ONLY or 'none (all)'}.")


In [ ]:
# =============================================================================
# 1. THE TILES: any tile with address points or with a flood band
# =============================================================================

POINT_COLS = ["cell_key", "people", "households", "over65", "imd_score", "imd_decile", "point_source",
              "rofrs_rank", "fz_rank", "fz_sea_rank", "fz_river_rank", "fz_river_and_sea_rank"]
points_t = spark.table(config.TBL_ADDRESS_POINTS)
hazard_t = spark.table(config.TBL_HAZARD_CELLS).filter(F.col("source").isin(["rofrs", "flood_zones", "flood_zones_sea", "flood_zones_river", "flood_zones_river_and_sea"]))

tiles_p = set(points_t.select("tile").distinct().toPandas()["tile"].dropna())
tiles_h = set(hazard_t.select("tile").distinct().toPandas()["tile"].dropna())
all_tiles = sorted((tiles_p | tiles_h) - {""})
print(f"{len(tiles_p)} tiles with address points, {len(tiles_h)} with a flood band, {len(all_tiles)} together.")
only_one = sorted(tiles_h - tiles_p), sorted(tiles_p - tiles_h)
print("Flood band but no points (sea or empty land):", only_one[0][:10], "| Points but no flood band:", only_one[1][:10])
tiles = [t for t in all_tiles if (not ONLY or t in ONLY)]
if ONLY and not tiles:
    raise ValueError(f"None of the tiles {ONLY} exist. Tiles are 50 km squares named like SE00, SE05, SE50, SE55 "
                     f"(two letters, then 00, 05, 50 or 55). Tiles available: {all_tiles}")

# critical sites, if job 31 has run
critical_cells = None
if USE_CRITICAL and spark.catalog.tableExists(config.TBL_CRITICAL_SITES):
    sites = spark.table(config.TBL_CRITICAL_SITES).toPandas()
    critical_cells = critical_to_cells(sites)
    cx, cy = _g.cell_xy(critical_cells["cell_key"].to_numpy())
    critical_cells["tile"] = tile_code_at(cx + 50.0, cy + 50.0)
    print(f"Critical sites: {len(sites):,} in {len(critical_cells):,} cells.")
else:
    print("No critical sites table (job 31 has not run): the critical site columns will be zero.")


In [ ]:
# =============================================================================
# 2. BUILD EACH TILE
# =============================================================================

results, log = [], []


def do_tile(tile, i, n):
    t0 = time.time()
    try:
        p = points_t.filter(F.col("tile") == tile).select(*POINT_COLS).toPandas()
        people = points_to_cells(p) if len(p) else pd.DataFrame({"cell_key": np.array([], dtype="int64")})
        long = hazard_t.filter(F.col("tile") == tile).select("source", "band", "cell_key", "share").toPandas()
        shares = shares_wide(long)
        crit = critical_cells[critical_cells["tile"] == tile].drop(columns="tile") if critical_cells is not None else None
        g = build_grid(people, shares, crit)
        g["tile"] = tile
        results.append(g)
        log.append({"tile": tile, "status": "done", "points": len(p), "cells": len(g), "seconds": round(time.time() - t0, 1)})
        print(f"[{i}/{n}] {tile}: {len(p):,} points, {len(g):,} cells, {time.time() - t0:.0f}s")
    except Exception as e:
        log.append({"tile": tile, "status": "FAILED: " + str(e)[:150]})
        print(f"[{i}/{n}] {tile}: FAILED {str(e)[:150]}")


work = [(t, i, len(tiles)) for i, t in enumerate(tiles, 1)]
if WORKERS == 1:
    for w in work:
        do_tile(*w)
else:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for f in as_completed([pool.submit(do_tile, *w) for w in work]):
            f.result()

failed = [r for r in log if r["status"].startswith("FAILED")]
print(f"\nTiles built: {len(results)}. Failed: {len(failed)}.")
if failed:
    display(pd.DataFrame(failed).astype(str))


In [ ]:
# =============================================================================
# 3. THE NATIONAL PERCENTILE, THEN SCORE AND WRITE EACH TILE ON ITS OWN
# The population percentile has to be national, so the people of every cell are gathered first (one column: small). Each tile is then scored
# against them and written, so the whole grid (several million rows, about 55 columns) is never joined or copied in one go: doing that
# exhausted the driver. A trial (tiles set) shows the result but does not write the table, because its percentile would not be national.
# =============================================================================

if failed:
    print("Not writing: some tiles failed. Re-run the failed tiles first.")
    results = []
all_people = (np.concatenate([r["people"].to_numpy(dtype="float64") if "people" in r else np.zeros(len(r)) for r in results])   # a sea tile has no people column
                if results else np.array([0.0]))
print(f"{len(all_people):,} cells in {len(results)} tiles. Score weights {config.RISK_SCORE_WEIGHTS}; surface water in the score: {config.RISK_SCORE_INCLUDE_SURFACE}.")

KEEP = (["cell_key", "easting", "northing", "tile", "people", "households", "over65", "imd_decile", "haz_rs", "critical_sites", "risk_score",
         "people_fz2", "people_fz3", "people_fz3_sea", "people_fz3_river", "people_fz3_both", "share_fz2", "share_fz3"]
        + [f"{p}_rs_{b}" for p in ("people", "households", "share") for b in ("very_low", "low", "medium", "high")])
slim, first = [], True
write = bool(results) and not ONLY and not failed
# every tile is written with the same columns in the same order: a tile with no address points (open sea) has no people columns at all
ALL_COLS = list(dict.fromkeys(c for r in results for c in r.columns))
KEYS_INT = ("cell_key", "easting", "northing")
if ONLY:
    print("Trial run: the table is not written.")
for g in results:
    g = g.reindex(columns=ALL_COLS)
    g["tile"] = g["tile"].astype(str)
    for c in ALL_COLS:                       # cells with no people columns have no people; every measure is a double
        if c in KEYS_INT:
            g[c] = g[c].astype("int64")
        elif c not in ("tile", "imd_score", "imd_decile"):
            g[c] = g[c].fillna(0.0).astype("float64")
        elif c != "tile":
            g[c] = g[c].astype("float64")
    g = add_score(g, config.RISK_SCORE_WEIGHTS, config.RISK_SCORE_CRITICAL_CAP, config.RISK_SCORE_INCLUDE_SURFACE, all_people=all_people)
    slim.append(g[[c for c in KEEP if c in g]].copy())
    if write:
        mode = "overwrite" if first else "append"
        spark.createDataFrame(g).write.format("delta").mode(mode).option("overwriteSchema", "true" if first else "false").saveAsTable(config.TBL_EXPOSURE_GRID)
        first = False
    del g
results = []                                 # free the tile frames
grid = pd.concat(slim, ignore_index=True) if slim else pd.DataFrame()
del slim
if write:
    spark.sql(f"""COMMENT ON TABLE {config.TBL_EXPOSURE_GRID} IS 'The 100 m British National Grid exposure grid for England: address points added up into cells (Census 2021 shares, IoD2025), the share of each cell in each flood band (rivers and sea, Flood Zones split coastal and fluvial), critical sites where available, and a composite risk score from config.RISK_SCORE_WEIGHTS. Rivers and sea only: no surface water. Restricted: built from address-level data. Job 34.'""")
    print(f"Written {len(grid):,} cells to {config.TBL_EXPOSURE_GRID}.")


In [ ]:
# =============================================================================
# 4. CHECKS
# =============================================================================

pts_tot = (points_t.filter(F.col("tile").isin(tiles) if ONLY else F.lit(True)).agg(F.sum("people").alias("p"), F.sum("households").alias("h")).toPandas().iloc[0])
print(f"People: grid {grid['people'].sum():,.0f}, address points {pts_tot['p']:,.0f} (equal). Households: grid {grid['households'].sum():,.0f}, address points {pts_tot['h']:,.0f}.")

bands = ["very_low", "low", "medium", "high"]
rs = pd.DataFrame({"band": bands, "people": [grid[f"people_rs_{b}"].sum() for b in bands], "households": [grid[f"households_rs_{b}"].sum() for b in bands],
                   "area_km2": [grid[f"share_rs_{b}"].sum() * 0.01 if f"share_rs_{b}" in grid else 0 for b in bands]}).round(0)
print("\nRivers and sea, by band (people and households at the address; area from the cell shares):")
display(rs.astype(str))
fz = pd.DataFrame({"layer": ["Flood Zone 2 only", "Flood Zone 3", "FZ3 sea (coastal)", "FZ3 river (fluvial)", "FZ3 mapped together"],
                   "people": [grid["people_fz2"].sum(), grid["people_fz3"].sum(), grid["people_fz3_sea"].sum(), grid["people_fz3_river"].sum(), grid["people_fz3_both"].sum()]}).round(0)
print("Flood Zones (the three FZ3 splits can overlap, so they do not add to Flood Zone 3):")
display(fz.astype(str))

print("\nScore: cells with a score above 0:", int((grid["risk_score"] > 0).sum()), " distribution of the non-zero scores:")
display(grid.loc[grid["risk_score"] > 0, "risk_score"].describe().round(2).to_frame("risk_score").reset_index().astype(str))
top = grid.sort_values("risk_score", ascending=False).head(15)[["cell_key", "easting", "northing", "tile", "people", "imd_decile", "haz_rs", "critical_sites", "risk_score"]]
print("Highest scoring cells (do these look like places you would expect?):")
display(top.round(2).astype(str))


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success" if not failed else "finished with failed tiles")
